### 1 - Load documents and build the index 

In [20]:
import json
from minsearch import Index


def load_documents(path="documents.json"):
    """Load the documents produced by 01-fetch-files.ipynb."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_index(documents):
    """Create and fit a minsearch index (content=text, filename=keyword)."""
    index = Index(text_fields=["content"], keyword_fields=["filename"])
    index.fit(documents)
    return index


documents = load_documents()
index = build_index(documents)
len(documents)  # should print 72

72

### 2 - Create the LLM client

Uses an OpenAI-style client with `gpt-5.4-mini`. The API key is read from the environment / `.env` (never hard-code or commit it).

In [21]:
from openai import OpenAI

llm_client = OpenAI()

### 3 - Prompt building blocks



In [22]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

### 4 - The RAG class 

In [23]:
from dataclasses import dataclass


@dataclass
class RAGResult:
    """Holds the final answer and the token usage of the request."""
    answer: str
    usage: object


class RAG:
    """Self-contained RAG over documents with `filename` and `content`."""

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model="gpt-5.4-mini",
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        # Our index has only `content` (text) and `filename` (keyword),
        # so we do a plain search without FAQ-style boost/filter.
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        # Build context from our schema: filename + content.
        lines = []
        for doc in search_results:
            lines.append(doc["filename"])
            lines.append(doc["content"])
            lines.append("")
        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(question=query, context=context)

    def llm(self, prompt):
        # Return the FULL response object (not just text) to expose usage.
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]
        return self.llm_client.responses.create(
            model=self.model, input=input_messages
        )

    def rag(self, query):
        # Return both the answer text and the usage.
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return RAGResult(answer=response.output_text, usage=response.usage)

### 5 - Run the RAG over the Q3 query

In [24]:
rag = RAG(index=index, llm_client=llm_client, model="gpt-5.4-mini")

query = "How does the agentic loop keep calling the model until it stops?"
result = rag.rag(query)

print(result.answer)

It keeps calling the model inside a `while True` loop and checks whether the response contains any `function_call` items.

- If there **is** a function call, the code runs the tool, appends the tool result to the message history, and loops again.
- If there are **no** function calls in that turn, it `break`s out of the loop.

So the stop condition is: **no function calls returned by the model**.


### 6 - Answer to Q3: input (prompt) tokens

In [25]:
result.usage

ResponseUsage(input_tokens=7111, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7208)

In [26]:
usage = result.usage
input_tokens = getattr(usage, "input_tokens", None) or getattr(usage, "prompt_tokens", None)
input_tokens

7111